# Understanding RAG: The Basics

LLMs are great with language, but they only know what was in their training data. Ask one about your company's internal docs, last week's news, or a blog post it has never seen, and it will either admit it doesn't know or, worse, confidently make something up.

**Retrieval-Augmented Generation (RAG)** fixes this with a simple idea: *before* the model answers, find the text that's relevant to the question and paste it into the prompt. Think of it as turning a closed-book exam into an open-book one. The model doesn't have to memorise your data. It just has to read the right page.

```text
 INDEXING  (done once, ahead of time)

   Load docs ──▶ Split into chunks ──▶ Embed each chunk ──▶ Vector store
                                                                 │
 QUERY TIME  (every question)                                    │
                                                                 ▼
   Question ──▶ Embed question ──▶ Find the most similar chunks (top k)
                                                │
                                                ▼
                       Prompt = instructions + chunks + question
                                                │
                                                ▼
                                       LLM ──▶ Answer
```

This notebook builds that pipeline one piece at a time:

0. **Setup**: install packages, load API keys, pick models
1. **Intuition**: tokens, embeddings, and similarity on a toy example
2. **Indexing**: load a real blog post, split it, embed it, store it
3. **Retrieval**: find the chunks that match a question
4. **Generation**: give those chunks to an LLM and get a grounded answer
5. **The full chain**: wire everything into a single call

**Stack:** [LangChain](https://python.langchain.com) · Gemini for embeddings and chat · [Chroma](https://www.trychroma.com) as the vector store. The structure follows LangChain's *RAG From Scratch* series.

## 0. Setup

Install the dependencies. You only need to do this once per environment.

In [ ]:
%pip install -q langchain langchain-community langchain-text-splitters langchain-google-genai langchain-chroma tiktoken beautifulsoup4 python-dotenv numpy

API keys live in a `.env` file next to this notebook (copy `.env.example` to get started), **never in the notebook itself**. Notebooks get committed and pushed, and keys inside them leak.

LangSmith is optional. If `LANGCHAIN_API_KEY` is set, every chain run gets traced at [smith.langchain.com](https://smith.langchain.com), so you can see exactly what was sent to the model at each step.

In [1]:
import os
from getpass import getpass
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass("Gemini API key: ")

# Optional: trace every run in LangSmith
if os.environ.get("LANGCHAIN_API_KEY"):
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"

# Sent as the User-Agent header when WebBaseLoader fetches pages
os.environ.setdefault("USER_AGENT", "understanding-rag-notebook")

print("LangSmith tracing:", "on" if os.environ.get("LANGCHAIN_TRACING_V2") else "off")

LangSmith tracing: on


RAG uses **two different models** for two different jobs:

- the **embedding model** turns text into vectors and powers the search.
- the **chat model** reads the retrieved text and writes the answer.

`temperature=0` keeps the answers as deterministic as possible, which helps when you're comparing runs.

In [2]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI

embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash", temperature=0)

## 1. Building intuition

Before we touch a real document, let's look at the three building blocks of RAG with a toy example: one question and one "document" that answers it.

In [3]:
question = "What kinds of pets do I like?"
document = "My favorite pet is a cat."

### 1.1 Tokens

Models don't read characters or words. They read **tokens**, pieces of text that are usually a whole word or part of one. Context windows, API pricing, and the chunk sizes we'll pick later are all measured in tokens, so it helps to see what they look like.

> `cl100k_base` is OpenAI's tokenizer. Gemini splits text a little differently, but the idea is the same, and `tiktoken` is a convenient way to count.

In [4]:
import tiktoken

encoding = tiktoken.get_encoding("cl100k_base")

def num_tokens(text: str) -> int:
    """Returns the number of tokens in a text string."""
    return len(encoding.encode(text))

tokens = encoding.encode(question)
print(f"{len(tokens)} tokens:", [encoding.decode([t]) for t in tokens])

8 tokens: ['What', ' kinds', ' of', ' pets', ' do', ' I', ' like', '?']


### 1.2 Embeddings

An **embedding** is a list of numbers that represents the *meaning* of a piece of text: a point in a space with thousands of dimensions. The embedding model is trained so that texts with similar meanings land close together.

A single number in the vector means nothing by itself. All that matters is how close two vectors are.

In [5]:
query_vector = embeddings.embed_query(question)
document_vector = embeddings.embed_documents([document])[0]

print("dimensions:", len(query_vector))
print("first 5 values:", [round(x, 4) for x in query_vector[:5]])

dimensions: 3072
first 5 values: [0.0081, -0.0115, 0.0173, -0.0486, 0.0062]


Notice we used `embed_query` for the question and `embed_documents` for the document. Gemini embeds them in slightly different modes (`RETRIEVAL_QUERY` vs `RETRIEVAL_DOCUMENT`), because a question and the passage that answers it are usually worded very differently. "What kinds of pets do I like?" looks nothing like "My favorite pet is a cat.", but we want them to match.

### 1.3 Cosine similarity

How do we measure "close"? The usual choice is **cosine similarity**, the cosine of the angle between two vectors:

$$\text{similarity}(A, B) = \cos(\theta) = \frac{A \cdot B}{\|A\| \, \|B\|}$$

- **close to 1**: the vectors point the same way, so the meanings are similar
- **close to 0**: the texts are unrelated

It only looks at direction, not length, so a short sentence and a long paragraph about the same thing can still score as similar.

In [6]:
import numpy as np

def cosine_similarity(vec1, vec2):
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

print(f"similarity: {cosine_similarity(query_vector, document_vector):.3f}")

similarity: 0.783


A single score doesn't tell you much. It gets interesting when we compare the question against **several** candidates and rank them. That ranking is what retrieval is:

In [7]:
candidates = [
    "My favorite pet is a cat.",
    "I adore my golden retriever, we go to the park every day.",
    "I'm allergic to fur, so animals aren't really my thing.",
    "The stock market closed higher today.",
    "Python is a popular programming language.",
]

candidate_vectors = embeddings.embed_documents(candidates)
scores = [cosine_similarity(query_vector, v) for v in candidate_vectors]

for score, text in sorted(zip(scores, candidates), reverse=True):
    print(f"{score:.3f}  {text}")

0.783  My favorite pet is a cat.
0.753  I'm allergic to fur, so animals aren't really my thing.
0.709  I adore my golden retriever, we go to the park every day.
0.594  Python is a popular programming language.
0.543  The stock market closed higher today.


A few things worth noticing:

- **Meaning beats keywords.** The golden-retriever sentence shares no meaningful words with the question, yet it scores well above the finance and programming sentences.
- **Embeddings capture topic, not stance.** The allergy sentence is about *not* wanting pets, yet it ranks **second**, above the golden retriever, because it's still about pets and animals. Retrieval finds text that is *related*. Working out what that text actually says is the LLM's job.
- **Scores are relative.** Even the unrelated sentences land around 0.55, nowhere near 0. With this model, the ranking and the gaps between scores matter, not the absolute number. That's one reason retrieval usually takes the "top k" instead of applying a fixed similarity cutoff.

**That's the heart of retrieval:** embed the question, score it against every stored chunk, and return the top *k*. Everything else in this notebook is plumbing to do that over real documents.

## 2. Indexing: load → split → embed → store

Indexing happens **once**, before anyone asks a question. Our knowledge base is Lilian Weng's post [*LLM Powered Autonomous Agents*](https://lilianweng.github.io/posts/2023-06-23-agent/).

### 2.1 Load

`WebBaseLoader` fetches the page. `SoupStrainer` tells BeautifulSoup to keep only the article (title, header, body) and skip the navigation, footer, and everything else.

In [8]:
import bs4
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
blog_docs = loader.load()

post = blog_docs[0].page_content
print(f"{len(blog_docs)} document · {len(post):,} characters · {num_tokens(post):,} tokens\n")
print(post[:400], "…")

C:\Users\risha\AppData\Local\Temp\ipykernel_15496\538410782.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader


1 document · 43,047 characters · 9,411 tokens



      LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, e …


### 2.2 Split

Why not just embed the whole post as one vector?

1. **Precision.** One embedding for a post this long is roughly the *average* of everything in it: planning, memory, tool use, case studies. A question about one topic would match all of it weakly. Small chunks each have a sharp, specific meaning.
2. **Context budget.** We only want to paste the *relevant* parts into the prompt, not the whole post.

`RecursiveCharacterTextSplitter` tries to break on paragraphs first, then lines, then words, so chunks end at natural boundaries.

- `chunk_size=300`: each chunk is at most ~300 tokens
- `chunk_overlap=50`: neighbouring chunks share up to ~50 tokens, so an idea that straddles a boundary still shows up whole in at least one chunk

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=300,
    chunk_overlap=50,
)
splits = text_splitter.split_documents(blog_docs)

sizes = [num_tokens(s.page_content) for s in splits]
print(f"{len(splits)} chunks · {min(sizes)}–{max(sizes)} tokens each (avg {sum(sizes) // len(sizes)})")

50 chunks · 4–285 tokens each (avg 196)


Here's the overlap in action. It only shows up where the splitter had to cut through the middle of a section. When a chunk ends on a clean paragraph break, there's nothing to repeat.

In [10]:
# Find the first pair of neighbouring chunks that share text
i = next(i for i in range(len(splits) - 1)
         if splits[i + 1].page_content[:60] in splits[i].page_content)

print(f"…end of chunk {i}:\n{splits[i].page_content[-250:]}\n")
print(f"start of chunk {i + 1}:\n{splits[i + 1].page_content[:250]}…")

…end of chunk 3:
ables LLM to interact with the environment (e.g. use Wikipedia search API), while the latter prompting LLM to generate reasoning traces in natural language.
The ReAct prompt template incorporates explicit steps for LLM to think, roughly formatted as:

start of chunk 4:
The ReAct prompt template incorporates explicit steps for LLM to think, roughly formatted as:
Thought: ...
Action: ...
Observation: ...
... (Repeated many times)…


### 2.3 Embed and store

Now we embed every chunk and store the vectors in a **vector store**, a database built to answer one question fast: *which stored vectors are closest to this one?*

Adding documents *appends* to a collection, so we reset it first. Otherwise every re-run of this cell would store each chunk again and retrieval would start returning duplicates. (Pass `persist_directory="./chroma_db"` if you'd like the index saved to disk between sessions.)

In [11]:
from langchain_chroma import Chroma

vectorstore = Chroma(collection_name="rag_from_scratch", embedding_function=embeddings)
vectorstore.reset_collection()  # start empty so re-running this cell doesn't duplicate chunks
vectorstore.add_documents(splits)

print(vectorstore._collection.count(), "chunks indexed")

50 chunks indexed


## 3. Retrieval

With the index built, retrieval is what we did by hand in 1.3: embed the question, compare it with every chunk, and keep the closest *k*.

Let's look at the scores first. Chroma reports a **distance** rather than a similarity, so **lower means closer**.

In [12]:
results = vectorstore.similarity_search_with_score("What is Task Decomposition?", k=4)

for doc, distance in results:
    preview = " ".join(doc.page_content.split())[:90]
    print(f"{distance:.3f}  {preview}…")

0.643  Component One: Planning# A complicated task usually involves many steps. An agent needs to…
0.686  Illustration of how HuggingGPT works. (Image source: Shen et al. 2023) The system comprise…
0.692  (2) Model selection: LLM distributes the tasks to expert models, where the request is fram…
0.707  10. Append to file: "append_to_file", args: "file": "<file>", "text": "<text>" 11. Delete …


Only the first hit is really about task decomposition. #2 and #3 describe HuggingGPT's pipeline, which also splits requests into tasks (related, but not what we asked). #4 is AutoGPT's list of commands, which most likely matched on all its "task" wording.

**Retrieval always returns k results, even when fewer than k are relevant.** The LLM has to cope with that noise, and reducing it is a big part of what advanced RAG techniques do.

A **retriever** wraps the vector store in the simplest interface possible: question in, documents out. That's what we'll plug into the chain.

In [13]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

docs = retriever.invoke("What is Task Decomposition?")
print(len(docs), "documents retrieved\n")
print(docs[0].page_content)

4 documents retrieved

Component One: Planning#
A complicated task usually involves many steps. An agent needs to know what they are and plan ahead.
Task Decomposition#
Chain of thought (CoT; Wei et al. 2022) has become a standard prompting technique for enhancing model performance on complex tasks. The model is instructed to “think step by step” to utilize more test-time computation to decompose hard tasks into smaller and simpler steps. CoT transforms big tasks into multiple manageable tasks and shed lights into an interpretation of the model’s thinking process.
Tree of Thoughts (Yao et al. 2023) extends CoT by exploring multiple reasoning possibilities at each step. It first decomposes the problem into multiple thought steps and generates multiple thoughts per step, creating a tree structure. The search process can be BFS (breadth-first search) or DFS (depth-first search) with each state evaluated by a classifier (via a prompt) or majority vote.
Task decomposition can be done (1) by

## 4. Generation

### 4.1 First, without retrieval

What does the model say when the answer isn't in its training data? Back to our toy question:

In [14]:
print(llm.invoke(question).text)

Since I don't have access to your personal memory or past conversations, I don't know yet! 

Tell me a little about yourself—or give me a few clues—and I can try to guess! Are you more of a dog person, a cat person, or do you prefer something a bit more unusual like reptiles, birds, or aquatic pets?


It can't know. Your pet preferences live in a document, not in the model's weights. That's exactly the gap RAG fills.

> **A trap when testing RAG:** ask about Lilian Weng's post *without* any context and the model often gets it right anyway, because the post is famous enough to be in its training data. To check that retrieval is really doing the work, test with information the model can't have memorised: private, recent, or made up.

### 4.2 The prompt

The "augmented" part of RAG is nothing more than a prompt template with a slot for the retrieved text. Telling the model to answer **based only on the following context** keeps it grounded in our document instead of whatever it memorised during training.

In [15]:
from langchain_core.prompts import ChatPromptTemplate

template = """Answer the question based only on the following context:
{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

The retriever returns `Document` objects. If we passed them straight into the prompt, the model would read their Python repr (`Document(metadata={...}, page_content='...')`): extra noise and wasted tokens. `format_docs` joins just the text.

Here's **exactly** what the model will receive. There's no magic: it's just a long string.

In [16]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

specific_question = "According to the post, what are the three ways task decomposition can be done?"
filled = prompt.invoke({"context": format_docs(docs), "question": specific_question}).to_string()

print(f"[{num_tokens(filled)} tokens]\n")
print(filled[:700], "\n\n[…]\n\n", filled[-150:], sep="")

[1068 tokens]

Human: Answer the question based only on the following context:
Component One: Planning#
A complicated task usually involves many steps. An agent needs to know what they are and plan ahead.
Task Decomposition#
Chain of thought (CoT; Wei et al. 2022) has become a standard prompting technique for enhancing model performance on complex tasks. The model is instructed to “think step by step” to utilize more test-time computation to decompose hard tasks into smaller and simpler steps. CoT transforms big tasks into multiple manageable tasks and shed lights into an interpretation of the model’s thinking process.
Tree of Thoughts (Yao et al. 2023) extends CoT by exploring multiple reasoning possibili

[…]

ete (Shutdown): "task_complete", args: "reason": "<reason>"

Question: According to the post, what are the three ways task decomposition can be done?



### 4.3 Ask with context

`|` pipes one step's output into the next. This is the *LangChain Expression Language* (LCEL). `StrOutputParser` pulls the plain text out of the model's response message.

Let's start with the toy example and give the model our one-sentence document as context:

In [17]:
from langchain_core.output_parsers import StrOutputParser

chain = prompt | llm | StrOutputParser()

print(chain.invoke({"context": document, "question": question}))

Based on the provided context, you like cats (specifically, a cat is your favorite pet).


Same model, same question. With one sentence of context, it goes from "I don't know you" to "cats". Now the real thing, using the chunks the retriever found:

In [18]:
print(chain.invoke({"context": format_docs(docs), "question": specific_question}))

Based on the provided context, task decomposition can be done in the following three ways:

1. **By LLM with simple prompting** (e.g., `"Steps for XYZ.\n1."` or `"What are the subgoals for achieving XYZ?"`)
2. **By using task-specific instructions** (e.g., `"Write a story outline."` for writing a novel)
3. **With human inputs**


The three ways match the retrieved chunk almost word for word (see the last paragraph of the chunk printed in section 3). The answer is grounded in a source we control, and we can point to exactly where it came from.

## 5. The full RAG chain

So far we've run retrieval and generation as separate steps. A RAG chain wires them together so a single question string flows through everything:

```text
                 ┌──▶ retriever ──▶ format_docs ──▶ context  ─┐
  "question" ────┤                                            ├──▶ prompt ──▶ llm ──▶ StrOutputParser ──▶ answer
                 └──▶ RunnablePassthrough ────────▶ question ─┘
```

The dict at the start runs both branches on the **same input**. The question is sent to the retriever to fetch context, and it's also passed through unchanged to fill `{question}`.

This time we use the popular [`rlm/rag-prompt`](https://smith.langchain.com/hub/rlm/rag-prompt) from the LangChain Hub. It's inlined below so the notebook doesn't need a Hub account, but you could also load it with `langsmith.Client().pull_prompt("rlm/rag-prompt")`. It adds two useful instructions: admit when you don't know, and keep it short.

In [19]:
from langchain_core.runnables import RunnablePassthrough

rag_prompt = ChatPromptTemplate.from_template(
    "You are an assistant for question-answering tasks. Use the following pieces of retrieved context "
    "to answer the question. If you don't know the answer, just say that you don't know. "
    "Use three sentences maximum and keep the answer concise.\n"
    "Question: {question} \nContext: {context} \nAnswer:"
)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

print(rag_chain.invoke("What is Task Decomposition?"))

Task decomposition is the process of breaking down complex tasks into smaller, simpler, and more manageable steps. It can be accomplished through methods such as Chain of Thought prompting (instructing the model to "think step by step"), task-specific instructions, or human inputs. This allows an agent or model to plan ahead and effectively tackle complicated tasks.


And a question the post has nothing to say about:

In [20]:
print(rag_chain.invoke("How do I bake sourdough bread?"))

I don't know. The provided context does not contain any information about how to bake sourdough bread.


The "if you don't know, say so" instruction does its job. Nothing relevant was retrieved, so the model declines instead of falling back on its general knowledge. For a system that's meant to answer from *your* documents, that's usually what you want.

## Recap

| Step | What it does | Here |
|---|---|---|
| **Load** | Get raw text from a source | `WebBaseLoader` |
| **Split** | Cut it into small, focused chunks | `RecursiveCharacterTextSplitter` (300 tokens, 50 overlap) |
| **Embed** | Turn each chunk into a vector that captures its meaning | `gemini-embedding-001` |
| **Store** | Index the vectors for fast nearest-neighbour search | Chroma |
| **Retrieve** | Embed the question, return the *k* closest chunks | `vectorstore.as_retriever()` |
| **Generate** | Paste the chunks into a prompt and let the LLM answer | `prompt \| llm \| StrOutputParser()` |

## Where basic RAG falls short

This pipeline works, but each step has an obvious weak spot, and the more advanced RAG techniques each target one of them:

- **Questions are often worded unlike the answer.** Rewriting the query, generating several versions (multi-query), or embedding a hypothetical answer (HyDE) improves recall.
- **Top-k by similarity isn't always the best k.** Re-ranking and fusing results from several searches (RAG-Fusion) help.
- **Chunk size is a trade-off.** Too small and you lose context; too big and you lose precision.
- **"It looks right" isn't evaluation.** You need a test set and metrics before you can tell whether a change helped.